## 1. Import libraries and set up the notebook

We import pandas for data preparation and Path for file paths.

We use datetime's timezone and timedelta to represent the fixed UTC+1 offset declared in the weather source.

We also import warnings and suppress warning messages in the notebook.

In [15]:
import pandas as pd
import warnings
from pathlib import Path
from datetime import timezone, timedelta
warnings.filterwarnings('ignore')

## 2. Read the raw weather file

We define the input path relative to the notebook location and load `weather-data_2025.csv` from the raw data folder.

We display the first rows to inspect the imported data.

In [16]:
# Define relative paths based on the notebook location (inside 'notebooks/' folder)
input_file = Path('../data/raw data/weather-data_2025.csv')

# Load the initial CSV file created in the previous preparation steps
df = pd.read_csv(input_file)

# Display the first few rows to verify
df.head()

,timestamp,temperature_2m,shortwave_radiation,Location
0,2025-01-01 00:00:00,-1.5,0.0,Lat:47.34622_Lon:8.543372
1,2025-01-01 01:00:00,-1.9,0.0,Lat:47.34622_Lon:8.543372
2,2025-01-01 02:00:00,-3.2,0.0,Lat:47.34622_Lon:8.543372
3,2025-01-01 03:00:00,-3.1,0.0,Lat:47.34622_Lon:8.543372
4,2025-01-01 04:00:00,-2.9,0.0,Lat:47.34622_Lon:8.543372


## 3. Convert timestamps to Swiss local time

The CSV timestamps have no timezone information. The original `weather-data_2025.json` declares `utc_offset_seconds: 3600`, and its hourly labels follow a continuous fixed UTC+1 clock, including the clock-change days.

We first assign this fixed source offset, then convert the same instants to Europe/Zurich to apply the Swiss winter and summer offsets.

For example, the source timestamp `2025-01-01 00:00:00` is already midnight in Zurich; interpreting it as UTC would incorrectly move it to 01:00.

We inspect the dataframe information to check the timestamp data type.

In [17]:
source_timezone = timezone(timedelta(hours=1))
df['timestamp'] = (
    pd.to_datetime(df['timestamp'])
    .dt.tz_localize(source_timezone)
    .dt.tz_convert('Europe/Zurich')
)

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 61320 entries, 0 to 61319
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype                        
---  ------               --------------  -----                        
 0   timestamp            61320 non-null  datetime64[us, Europe/Zurich]
 1   temperature_2m       61320 non-null  float64                      
 2   shortwave_radiation  61320 non-null  float64                      
 3   Location             61320 non-null  str                          
dtypes: datetime64[us, Europe/Zurich](1), float64(2), str(1)
memory usage: 3.4 MB


## 4. Select the 2025 study period

As in the energy preparation, we select rows by their year in Europe/Zurich after converting the timestamps.

We sort the observations chronologically and reset the index. This keeps the same local study period in both datasets.

In [18]:
df = (
    df.loc[df['timestamp'].dt.year == 2025]
    .sort_values('timestamp')
    .reset_index(drop=True)
)
print(f'Weather observations in 2025: {len(df):,}')
display(df.head(2), df.tail(2))


Weather observations in 2025: 61,320


,timestamp,temperature_2m,shortwave_radiation,Location
0,2025-01-01 00:00:00+01:00,-1.5,0.0,Lat:47.34622_Lon:8.543372
1,2025-01-01 00:00:00+01:00,-2.2,0.0,Lat:46.85413_Lon:9.46602


,timestamp,temperature_2m,shortwave_radiation,Location
61318,2025-12-31 23:00:00+01:00,-2.7,0.0,Lat:46.22144_Lon:6.172249
61319,2025-12-31 23:00:00+01:00,-3.2,0.0,Lat:46.85413_Lon:9.46602


## 5. Remove latitude and longitude columns

We remove the latitude and longitude columns if they are present in the dataframe.

We display the first rows to inspect the remaining columns.

In [19]:
# Safely drop 'latitude' and 'longitude' if they exist in the current dataframe
columns_to_drop = ['latitude', 'longitude']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

# Display updated columns
df.head()


,timestamp,temperature_2m,shortwave_radiation,Location
0,2025-01-01 00:00:00+01:00,-1.5,0.0,Lat:47.34622_Lon:8.543372
1,2025-01-01 00:00:00+01:00,-2.2,0.0,Lat:46.85413_Lon:9.46602
2,2025-01-01 00:00:00+01:00,-3.2,0.0,Lat:47.557117_Lon:7.549342
3,2025-01-01 00:00:00+01:00,3.1,0.0,Lat:46.010544_Lon:8.857143
4,2025-01-01 00:00:00+01:00,-0.4,0.0,Lat:46.22144_Lon:6.172249


## 6. Aggregate weather data by timestamp

We group the observations by timestamp and calculate the mean temperature and shortwave radiation for each timestamp.

We round the results to four decimal places and reset the index so that timestamp remains a column.

We display the first rows of the aggregated data.

In [20]:
# Group by timestamp, calculate the mean for specific columns, round to 4 decimals, and reset the index
df_avg = df.groupby('timestamp')[['temperature_2m', 'shortwave_radiation']].mean().round(4).reset_index()

# Display the aggregated results
df_avg.head()


,timestamp,temperature_2m,shortwave_radiation
0,2025-01-01 00:00:00+01:00,-1.4571,0.0
1,2025-01-01 01:00:00+01:00,-1.7857,0.0
2,2025-01-01 02:00:00+01:00,-2.4000,0.0
3,2025-01-01 03:00:00+01:00,-2.4714,0.0
4,2025-01-01 04:00:00+01:00,-2.5714,0.0


## 7. Check hourly coverage and measurements

After averaging the locations, we expect 8,760 distinct hourly instants in 2025, just as in the energy dataset.

We compare the timestamps with a timezone-aware hourly calendar to detect missing hours, unexpected timestamps and duplicates, including around the clock changes.

We also check that the aggregated temperature and shortwave radiation values are numeric and complete. Zero radiation is a valid measurement.

Any non-zero count stops execution before export.

In [21]:
expected = pd.date_range('2025-01-01', '2026-01-01', freq='h',
                         tz='Europe/Zurich', inclusive='left')
actual = pd.DatetimeIndex(df_avg['timestamp'])

for column in ['temperature_2m', 'shortwave_radiation']:
    df_avg[column] = pd.to_numeric(df_avg[column], errors='raise')

checks = pd.Series({
    'Missing expected hours': len(expected.difference(actual)),
    'Unexpected timestamps': len(actual.difference(expected)),
    'Duplicate timestamps': int(actual.duplicated().sum()),
    'Missing temperature values': int(df_avg['temperature_2m'].isna().sum()),
    'Missing radiation values': int(df_avg['shortwave_radiation'].isna().sum()),
})
display(checks.to_frame('Count'))
if checks.any():
    raise ValueError('Review the non-zero counts before exporting.')


,Count
Missing expected hours,0
Unexpected timestamps,0
Duplicate timestamps,0
Missing temperature values,0
Missing radiation values,0


## 8. Export the prepared weather data

The coverage checks confirm that the timestamps represent the complete 2025 study period in Europe/Zurich.

We format the timestamps as ISO 8601 strings with seconds and an explicit offset, for example `2025-01-01T00:00:00+01:00`.

We ensure the processed data folder exists and export the aggregated data to `weather-data_2025.csv` without the dataframe index.

We print the output path to confirm where the file was saved.

In [22]:
# Wandelt den Zeitstempel exakt in das Format YYYY-MM-DDTHH:MM:SS+01:00 um
df_avg['timestamp'] = df_avg['timestamp'].apply(lambda x: x.isoformat(timespec='seconds'))

# Export the cleaned and aggregated dataframe to a new CSV file in the 'clean data' folder
output_file = Path('../data/processed data/weather-data_2025.csv')

# Ensure the output directory exists
output_file.parent.mkdir(parents=True, exist_ok=True)

df_avg.to_csv(output_file, index=False)

print(f'Data successfully saved to {output_file}')

Data successfully saved to ../data/processed data/weather-data_2025.csv
